# Mass Spectrometry Group Alignment of Mouse Pancreate Data

Dataset can be downloaded from https://zenodo.org/record/3607915#.Y_To9nbMJrr and paper from doi: https://doi.org/10.1016/j.molmet.2020.01.017.

## Preparation

Loading Packages

In [ ]:
import os,csv,random
import pandas as pd
import numpy as np
import scanpy as sc
import math

from skimage import io, color
import torch


In [ ]:
from scanpy import read_10x_h5
import SpaGCN as spg
import matplotlib.pyplot as plt
import json

In [ ]:
from tqdm import tqdm
import pickle

In [ ]:
from pyimzml.ImzMLParser import ImzMLParser
from pyimzml.metadata import Metadata

In [ ]:
# === CONFIGURE ME ===
DATA_DIR = 'data'        # input data root (subdirectories per dataset live underneath)
OUTPUT_DIR = 'output'    # where this notebook writes its outputs
REPO_ROOT = '.'                # any remaining absolute-path references resolve to here
# ====================


In [ ]:
import GalaxyPython as gx
gx.__version__

## Data Loading

We first take the DS26 (the normal tissue) as the reference 

In [ ]:
Ref_file = f'{DATA_DIR}/MousePancreate/mouse_islet_of_langerhans_preliminary_peaks.imzML'
p = ImzMLParser(Ref_file)
spectra_mz = []
spectra_intensity = []
x_cord = []
y_cord = []
z_cord = []
for idx, (x,y,z) in enumerate(p.coordinates):
    mzs, intensities = p.getspectrum(idx)
    spectra_mz.append(mzs)
    spectra_intensity.append(intensities)
    x_cord.append(x)
    y_cord.append(y)
    z_cord.append(z)

In [ ]:
plt.scatter(mzs, intensities, alpha=0.5)
plt.show()

In [ ]:
len(spectra_mz)

## Data Preprocessing
skip this step if .h5ad is already generated.
#### Gaussian Kernal Weighted Average by Nadaraya–Watson estimator

Due to the difference between to  m/z values of two adjacent observation is not evenly, we employed Gaussian Kernal to grid the processed m/z values into grid. i.e., the values are evenly distributed. 

In [ ]:
RESULTS = gx.gridding(mzs, intensities)
RESULTS

Zero percentage:

In [ ]:
np.count_nonzero(RESULTS[1])/RESULTS[1].shape[0]

In [ ]:
plt.scatter(RESULTS[0], RESULTS[1], alpha=0.5)
plt.show()

#### Grid all m/z values to the same reference grid.

In [ ]:
STARTmz = 70
ENDmz = 600

spectra_gridded = []
for idx in tqdm(range(len(spectra_mz))):
    RESULTS = gx.gridding(spectra_mz[idx], spectra_intensity[idx], start = STARTmz, end = ENDmz, increment = 0.25)
    spectra_gridded.append(RESULTS[1])
    
    

In [ ]:
mzs_new = RESULTS[0]

In [ ]:
mz_value = pd.DataFrame(mzs_new, index = mzs_new.astype('float'))
mz_value = mz_value.rename(columns = {0:"m/z"})

In [ ]:
MALDIloc = pd.DataFrame({'x':x_cord, 'y':y_cord, 'z':z_cord})

In [ ]:
MALDIspectrum = np.stack(spectra_gridded)    
MALDIdataAnn = sc.AnnData(X = MALDIspectrum, var = mz_value, obs = MALDIloc)

In [ ]:
MALDIdataAnn.write_h5ad(filename = f"{DATA_DIR}/MousePancreate/MousePancreate_small.h5ad")

## Simulation

Load the Annotated data directly

In [ ]:
MALDIdataAnn = sc.read_h5ad (filename  = f"{DATA_DIR}/MousePancreate/MousePancreate_small.h5ad")

In [ ]:
MALDIdataAnn.obs = MALDIdataAnn.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn)

### 0. Preparation: peaks calling and peaks grouping

In [ ]:
PeakGroup = gx.PeakCalling_single (MALDIdataAnn)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.95)

We first create an MALDIdataAnn object of the exact alignment of two Mass Spectrometry data. Since it is expected to be an exact alignment, we consider the unknown and reference data are from the same dataset.

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn, MALDIdataAnn)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.8, ignore = True)
ExactAlign.summarize()

The results show that no shifting adjustment need to be done as the shift lengths for all groups are 0. 

In [ ]:
ExactAlign.changerecord

Define the function of conducting simulations studies.

In [ ]:
def SIM ( change_trace_first12, sigma, filename):
    ExactAlign.changerecord[0:12] = change_trace_first12
    SD = gx.MALDI_SIM(ExactAlign)
    MALDISimData = SD.getAnnSim(shuffle = True, sigma = 0.1, nregion = 1)
    # Stage 1: Peak grouping ...
    Stage1 = gx.PeakCalling (MALDISimData, MALDIdataAnn)
    Stage1.peak_calling(0.9)
    Stage1.peak_grouping(0.95)
    # Stage 2: Alignment ...
    Stage2 = gx.AnnDataMALDI(MALDISimData, MALDIdataAnn)
    Stage2.get_corr_peakgroup_refined(Stage1.jointcluster)
    Stage2.peak_group_pairing()
    Stage2.fine_alignment_assessment(threshold= 0.2, ignore = True)
    Stage2.summarize()
    # Simulation Done. The results:
    results = SD.MSE(Stage2.unknownalign, Stage2.referenalign)
    # MALDISimData.write(filename = filename)
    return results[0].mean()

In [ ]:
MALDIdataAnn.write(filename = f"{OUTPUT_DIR}/simulation/simdata/SimData_Ref.h5ad")

## MS-GAlign

### 1. Scenario 1: small changes (sigma = 0.2)

In [ ]:
random.seed(2023)
MSE = []

for i in range(500):
    try:
        results = SIM ([1,0,2,0,1,0,0,0,0,0,0,0], sigma = 0.2, filename = f"{OUTPUT_DIR}/simulation/simdata/SimData_{index}.h5ad".format(index = i))
        MSE.append(results)
    except:
        MSE.append(float('nan'))
    
   

In [ ]:
np.savetxt(f'{OUTPUT_DIR}/simulation/small_changes_sigma01.txt', np.array(MSE), delimiter=',')

In [ ]:
np.array(MSE)

In [ ]:
SD = gx.MALDI_SIM(ExactAlign)
MALDISimData = SD.getAnnSim(shuffle = True, sigma = 0.1, nregion = 1)

In [ ]:
SD.shiftdatadf

Stage 1: Peak grouping ...

In [ ]:
Stage1 = gx.PeakCalling (MALDISimData, MALDIdataAnn)
Stage1.peak_calling(0.9)
Stage1.peak_grouping(0.95)

Stage 2: Alignment ...

In [ ]:
Stage2 = gx.AnnDataMALDI(MALDISimData, MALDIdataAnn)

In [ ]:
Stage2.meanspectrumUnk[110:140]

In [ ]:
Stage2.meanspectrumRef[110:140]

In [ ]:
Stage2 = gx.AnnDataMALDI(MALDISimData, MALDIdataAnn)
Stage2.get_corr_peakgroup_refined(Stage1.jointcluster)
Stage2.peak_group_pairing()
Stage2.fine_alignment_assessment(threshold= 0.2, ignore = True)
Stage2.summarize()

Simulation Done. The results:

In [ ]:
results = SD.MSE(Stage2.unknownalign, Stage2.referenalign)

Output the results:

In [ ]:
results[0].sum()

### 2. Scenario 2: large changes (sigma = 0.5)

In [ ]:
def SIM_large ( change_trace_first12, sigma, filename):
    ExactAlign.changerecord = change_trace_first12
    SD = gx.MALDI_SIM(ExactAlign)
    MALDISimData = SD.getAnnSim(shuffle = True, sigma = 0.1, nregion = 2)
    # Stage 1: Peak grouping ...
    Stage1 = gx.PeakCalling (MALDISimData, MALDIdataAnn)
    Stage1.peak_calling(0.9)
    Stage1.peak_grouping(0.95)
    # Stage 2: Alignment ...
    Stage2 = gx.AnnDataMALDI(MALDISimData, MALDIdataAnn)
    Stage2.get_corr_peakgroup_refined(Stage1.jointcluster)
    Stage2.peak_group_pairing()
    Stage2.fine_alignment_assessment(threshold= 0.2, ignore = True)
    Stage2.summarize()
    # Simulation Done. The results:
    results = SD.MSE(Stage2.unknownalign, Stage2.referenalign)
    # MALDISimData.write(filename = filename)
    return results[0].mean()

In [ ]:
random.seed(2023)
MSE = []

for i in range(500):
    try:
        results = SIM_large ([1,2,1,0,1,0,0, 0, 0, 0, 0, 0, 0, 0, 0, 0,3,0,2,0,1], sigma = 0.5, filename = f"{OUTPUT_DIR}/simulation/simdata/SimData_large_{index}.h5ad".format(index = i))
        MSE.append(results)
    except:
        MSE.append(float('nan'))

In [ ]:
np.savetxt(f'{OUTPUT_DIR}/simulation/large_changes_sigma05.txt', np.array(MSE), delimiter=',')

In [ ]:
MSE

In [ ]:
data = pd.read_csv(f"{OUTPUT_DIR}/simulation/large_changes_sigma05.txt", header = None)
data.mean()

In [ ]:
### 2. Scenario 3: large changes (sigma = 1)

In [ ]:
random.seed(2023)
MSE = []

for i in range(500):
    try:
        results = SIM ([1,2,1,0,1,3,0,2,0,2,1,0], sigma = 1, filename = f"{OUTPUT_DIR}/simulation/simdata/SimData_large_{index}.h5ad".format(index = i))
        MSE.append(results)
    except:
        MSE.append(float('nan'))

In [ ]:
np.savetxt(f'{OUTPUT_DIR}/simulation/large_changes_sigma01.txt', np.array(MSE), delimiter=',')

In [ ]:
MSE

In [ ]:
data = pd.read_csv(f"{OUTPUT_DIR}/simulation/large_changes_sigma01.txt", header = None)
data.mean()

## Competing method: MSIwarp

In [ ]:
import msiwarp as mx
from msiwarp.util.warp import to_mx_peaks
from msiwarp.util.warp import to_mz, to_height


In [ ]:
def MSIwarping (MALDIdataAnn1, MALDIdataAnn2):
    spectra = []
    mzs = np.array(MALDIdataAnn1.var["m/z"]).astype(np.float) # peak m/z values
    meanspectrum1 =  np.mean(MALDIdataAnn1.X, axis = 0)
    for i in range(0,MALDIdataAnn1.X.shape[0],100):
        hs = meanspectrum1
        spectra.append( [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs, hs))])
    #
    hs = meanspectrum1 # peak heights
    s = [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs, hs))]
    #
    mzs2 = np.array(MALDIdataAnn2.var["m/z"]).astype(np.float) # peak m/z values
    meanspectrum2 =  np.mean(MALDIdataAnn2.X, axis = 0)
    hs2 = meanspectrum2 # peak heights
    s2 = [mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs2, hs2))]
    #
    reference_spectrum =  s2
    ### 1.2 setup the node placement parameters ######
    # method 1
    node_mzs = [i for i in range(15,1200,50)]
    node_deltas = [i*0.01 for i in range(15,121,5)] # slacks = node_deltas * n_steps
    n_steps = 30
    nodes = mx.initialize_nodes(node_mzs, node_deltas, n_steps)
    epsilon = 1 # peak matching threshold, relative to peak width
    #
    optimal_moves = mx.find_optimal_spectra_warpings(spectra, reference_spectrum, nodes, epsilon)
    spectra2 = [[mx.peak(i, mz_i, h_i, 1.0) for i, (mz_i, h_i) in enumerate(zip(mzs.astype(np.float), mzs))]]
    #
    warped_spectra = [mx.warp_peaks(s_i, nodes, optimal_moves[0]) for s_i in spectra2]
    #
    newmz = to_mz(warped_spectra[0])
    oldmz = to_height(warped_spectra[0])
    return {"newmz":newmz, "oldmz":oldmz}

In [ ]:
len(orgmzidx)

In [ ]:
## MSE result plot
resultsdf = pd.DataFrame({"MSE":MSE, "unkmz":SD.truemz[orgmzidx], "refmz": warpresult.get("newmz")})
resultsdf.to_csv("output/simulation/MSE_results_1simulation_wapring.csv", sep = ",")

In [ ]:
def SIM_msiwarp (change_trace_first12, sigma):
    ExactAlign.changerecord[0:12] = change_trace_first12
    SD = gx.MALDI_SIM(ExactAlign)
    MALDISimData = SD.getAnnSim(shuffle = True, sigma = 0.1, nregion = 1)
    # Warping ...
    warpresult = MSIwarping (MALDISimData, MALDIdataAnn_REf)
    orgmz = warpresult.get("oldmz")
    orgmzidx = [np.where(abs(np.array(MALDISimData.var["m/z"])-orgmzi)==abs(np.array(MALDISimData.var["m/z"])-orgmzi).min())[0].max() for orgmzi in orgmz]
    # Simulation Done. The results:
    results = np.mean(np.square(SD.truemz[orgmzidx] - warpresult.get("newmz")))
    return results

In [ ]:
results = SIM_msiwarp ([1,0,2,0,1,0,0,0,0,0,0,0], sigma = 0.2)
results

### 1. Scenario 1: small changes (sigma = 0.2)

In [ ]:
random.seed(2023)
MSE = []

for i in range(500):
    try:
        results = SIM_msiwarp ([1,0,2,0,1,0,0,0,0,0,0,0], sigma = 0.2)
        MSE.append(results)
    except:
        MSE.append(float('nan'))


In [ ]:
np.savetxt(f'{OUTPUT_DIR}/simulation/small_changes_sigma02_MSIwarp.txt', np.array(MSE), delimiter=',')

In [ ]:
### 1. Scenario 2: Large changes (sigma = 0.2)